# Un transformer como modelo del mundo de CartPole

Notebook opcional del **módulo 18** del curso (en la app: `uv run dev` y la ruta `#/modulo/world-models`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

env = gym.make("CartPole-v1")
S, A, S2 = [], [], []
for ep in range(200):
    s, _ = env.reset(seed=ep)
    for _ in range(200):
        a = env.action_space.sample()
        s2, r, fin, trunc, _ = env.step(a)
        S.append(s)
        A.append(a)
        S2.append(s2)
        s = s2
        if fin or trunc:
            break
S, A, S2 = (torch.tensor(v, dtype=torch.float32) for v in (S, A, S2))
print(len(S), "transiciones")

In [ ]:
class ModeloMundo(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.inp = nn.Linear(5, d)
        capa = nn.TransformerEncoderLayer(d, 4, 128, batch_first=True, norm_first=True)
        self.tr = nn.TransformerEncoder(capa, 2)
        self.out = nn.Linear(d, 4)

    def forward(self, s, a):            # s: (B, L, 4), a: (B, L)
        x = self.inp(torch.cat([s, a[..., None]], -1))
        m = nn.Transformer.generate_square_subsequent_mask(x.shape[1])
        return s + self.out(self.tr(x, mask=m, is_causal=True))   # predice el cambio de estado


mm = ModeloMundo()
opt = torch.optim.Adam(mm.parameters(), 1e-3)
L = 8
for paso in range(1500):
    i = torch.randint(0, len(S) - L, (64,))
    s = torch.stack([S[j:j + L] for j in i])
    a = torch.stack([A[j:j + L] for j in i])
    s2 = torch.stack([S2[j:j + L] for j in i])
    perdida = ((mm(s, a) - s2) ** 2).mean()
    opt.zero_grad()
    perdida.backward()
    opt.step()
print("pérdida:", perdida.item())

In [ ]:
s, _ = env.reset(seed=999)
real, imaginado = [s], [torch.tensor(s, dtype=torch.float32)]
acciones = []
for t in range(15):
    a = env.action_space.sample()
    acciones.append(a)
    s, *_ = env.step(a)
    real.append(s)
    with torch.no_grad():
        pred = mm(torch.stack(imaginado)[None], torch.tensor(acciones, dtype=torch.float32)[None])[0, -1]
    imaginado.append(pred)
plt.plot([r[2] for r in real], label="ángulo real")
plt.plot([float(i[2]) for i in imaginado], "--", label="ángulo imaginado")
plt.legend()
plt.show()